# OGI Fabric — CD PDE Adaptive Control

**Corten Research — Singleton, Dollinger & Spence**  
**arXiv:2411.15832 Phase 2**

---

This notebook documents the adversarial stability test and adaptive goal anchor
developed during Phase 2 validation.

**Key finding:** The CD PDE state update is genuinely bounded under adversarial
pressure. However, local contraction does not imply persistent goal retention.
Without a feedback mechanism, the system can remain dynamically stable while
drifting semantically off-goal.

**The adaptive anchor** closes this gap with a single-step feedback correction:
measure alignment between PDE state and goal projection, increase correction
strength when alignment degrades.

**This is not yet a full control loop.** See the Next Steps cell at the end.

In [ ]:
# Cell 1 — Environment setup
import sys
import subprocess
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist

# Path
sys.path.insert(0, '/mnt/custom-file-systems/s3/shared/ogi_navi_sad')

# Install CD library
subprocess.run(
    ['pip', 'install', '-e', '/home/sagemaker-user/navi-creative-determinant', '--quiet'],
    capture_output=True
)

from cd import (
    solve_1d_picard,
    viability_canonical,
    gaussian_bump_1d,
    check_convergence,
    solution_type,
    viability_threshold_1d,
)
from cd_state_updater import FabricStateUpdater, CDStateUpdater, CD_AVAILABLE

print(f"CD available: {CD_AVAILABLE}")

In [ ]:
# Cell 2 — Load Fabric
%run ogi_fabric_standalone.ipynb

## v5: Adaptive Goal Anchor

Extends the v4 CD PDE update with a feedback correction term.

After the PDE solve, measure cosine alignment between the PDE solution
and the goal projection. If alignment is low, increase the correction
strength alpha toward the goal anchor.

```
boost = max(0.0, 0.2 - alignment)
alpha = min(max_alpha, base_alpha + boost)
new_state = (1 - alpha) * pde_state + alpha * goal_proj
```

Parameters:
- `base_alpha = 0.10` — baseline correction strength
- `max_alpha = 0.25`  — cap to prevent over-damping
- Threshold `0.2`     — alignment below this triggers boost

In [ ]:
# Cell 3 — Patch CDStateUpdater with v5 adaptive update

def _update_v5_adaptive(self, compressed, state, goal_query, step=0, module_id='?'):
    msg_np   = compressed.detach().numpy().flatten()
    query_np = goal_query.detach().numpy().flatten()

    N = self.N
    L = self.L
    x = np.linspace(0, L, N + 2)[1:-1]

    # Step 1: viability_canonical baseline (weak — bump dominates)
    m = msg_np[:N]
    lo, hi = m.min(), m.max()
    mu     = np.ones(N) * 0.3 if hi - lo < 1e-8 else (m - lo) / (hi - lo)
    b_base = viability_canonical(kappa=0.2, gamma=0.5, mu=mu, lam=0.1)

    # Step 2: goal-specific Gaussian bump
    q      = query_np
    q_norm = (q - q.min()) / (q.max() - q.min() + 1e-8)
    center    = float(0.1 + 0.8 * np.median(q_norm[:16]))
    amplitude = float(1.0 + 1.0 * np.std(q_norm))
    bump      = gaussian_bump_1d(x, center, sigma=0.12, amplitude=amplitude)
    b_field   = b_base + bump

    # Step 3: spectral calibration
    b_mean   = max(float(b_field.mean()), 1e-8)
    beta_b   = (np.pi**2 / L**2 - (-1.0)) / b_mean
    beta_arr = beta_b * b_field

    try:
        x_out, Phi, info = solve_1d_picard(
            L=L, N=N, a=0.0,
            beta_b=beta_arr, c=10.0,
            damping=0.4, max_iter=10000, tol=1e-9,
        )
        converged = check_convergence(info)[0]
        sol_type  = solution_type(info)

        Phi = Phi.flatten()
        if len(Phi) == N + 2:
            Phi = Phi[1:-1]
        Phi = Phi[:N]

        # Normalize PDE solution
        pde_state = torch.tensor(Phi, dtype=torch.float32).unsqueeze(0)
        pde_state = pde_state / (torch.norm(pde_state) + 1e-8)

        # Goal projection into state space
        goal_proj = torch.zeros_like(pde_state)
        w = min(goal_query.shape[1], pde_state.shape[1])
        goal_proj[:, :w] = goal_query[:, :w]
        goal_proj = goal_proj / (torch.norm(goal_proj) + 1e-8)

        # Adaptive anchor — feedback correction
        base_alpha = getattr(self, 'goal_anchor_alpha', 0.10)
        max_alpha  = getattr(self, 'goal_anchor_max',   0.25)
        alignment  = torch.nn.functional.cosine_similarity(
            pde_state, goal_proj, dim=1
        ).mean().item()
        boost     = max(0.0, 0.2 - alignment)
        alpha     = min(max_alpha, base_alpha + boost)
        new_state = (1.0 - alpha) * pde_state + alpha * goal_proj

    except Exception as e:
        print(f"Solver failed step={step} module={module_id}: {e}")
        new_state = state

    return new_state


CDStateUpdater.update = _update_v5_adaptive
print("Patched v5 — adaptive goal anchor active")

## Adversarial Stability Test

Test setup:
- T=50 steps of increasing adversarial pressure
- Message corruption grows linearly with step: `pressure = t * 0.04`
- Goal vector perturbed each step: `goal += noise * randn`
- Sweep alpha in {0.00, 0.05, 0.10, 0.20}

Metrics tracked:
- Step-wise drift: `||state_t - state_{t-1}||`
- Cumulative drift
- State norm
- Alignment: `cosine(state, goal_proj)`

In [ ]:
# Cell 4 — Adversarial test: alpha sweep

ALPHAS      = [0.00, 0.05, 0.10, 0.20]
T           = 50
PRESSURE    = 0.04
NOISE_SCALE = 0.02
GOAL_DRIFT  = 0.03
SEED        = 42
D_MSG       = 64
D_STATE     = 64
D_QUERY     = 32

torch.manual_seed(SEED)
np.random.seed(SEED)


def cosine_sim(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=1).mean().item()

def project_goal(goal, d_state):
    out = torch.zeros(goal.shape[0], d_state)
    w   = min(goal.shape[1], d_state)
    out[:, :w] = goal[:, :w]
    return out

def adversarial_message(step):
    base       = torch.randn(1, D_MSG)
    corruption = (step * PRESSURE) * torch.randn(1, D_MSG)
    return base + corruption


def run_experiment(alpha):
    updater = FabricStateUpdater(
        d_msg=D_MSG, d_state=D_STATE, d_query=D_QUERY, use_cd=True
    )
    updater.cd.goal_anchor_alpha = alpha
    updater.cd.goal_anchor_max   = 0.25

    state = torch.randn(1, D_STATE)
    goal  = torch.randn(1, D_QUERY)

    drift_history     = []
    cum_drift         = []
    norm_history      = []
    alignment_history = []
    D_total           = 0.0

    for t in range(T):
        pressure = t * PRESSURE
        noise    = NOISE_SCALE + pressure
        msg      = adversarial_message(t) + noise * torch.randn(1, D_MSG)
        goal     = goal + (GOAL_DRIFT + pressure) * torch.randn_like(goal)

        prev_state = state.clone()
        state      = updater(msg, state, goal, step=t, module_id='A')

        delta    = torch.norm(state - prev_state).item()
        D_total += delta
        drift_history.append(delta)
        cum_drift.append(D_total)
        norm_history.append(torch.norm(state).item())
        alignment_history.append(cosine_sim(state, project_goal(goal, D_STATE)))

    return {
        'alpha':          alpha,
        'drift':          drift_history,
        'cum_drift':      cum_drift,
        'norm':           norm_history,
        'alignment':      alignment_history,
        'final_drift':    D_total,
        'final_norm':     norm_history[-1],
        'final_alignment':alignment_history[-1],
    }


results = [run_experiment(a) for a in ALPHAS]

print("\n=== ADVERSARIAL TEST SUMMARY ===")
print(f"{'alpha':>8} {'cum_drift':>12} {'final_norm':>12} {'final_align':>13}")
print("-" * 50)
for r in results:
    print(f"  {r['alpha']:>6.2f} {r['final_drift']:>12.4f} "
          f"{r['final_norm']:>12.4f} {r['final_alignment']:>13.4f}")

In [ ]:
# Cell 5 — Visualization

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Adversarial Stability Test — Alpha Sweep', fontweight='bold')

for r in results:
    label = f"α={r['alpha']}"
    axes[0].plot(r['cum_drift'],  label=label)
    axes[1].plot(r['norm'],       label=label)
    axes[2].plot(r['alignment'],  label=label)

axes[0].set_title('Cumulative Drift');  axes[0].set_xlabel('Step')
axes[1].set_title('State Norm');        axes[1].set_xlabel('Step')
axes[2].set_title('Goal Alignment');    axes[2].set_xlabel('Step')
axes[2].axhline(0.0, color='gray', linestyle='--', alpha=0.5)

for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cd_adaptive_control_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cd_adaptive_control_sweep.png")

## Findings

**1. The PDE update is genuinely bounded.**  
Removing the norm clamp did not cause explosion or collapse. State norm
remains stable across all 50 adversarial steps. The contraction behavior
from MonotoneFixedPoint.lean is real.

**2. Local contraction does not imply persistent goal retention.**  
Without correction (alpha=0.00), the system remains dynamically stable
while alignment degrades under sustained pressure. Stable dynamics and
intelligent routing are not enough on their own.

**3. The adaptive anchor improves alignment.**  
Base alpha=0.10 with adaptive boost gives the best balance: alignment
stays positive under pressure, cumulative drift remains controlled,
state norm remains bounded.

**4. Even alpha=0.00 with adaptive boost prevents full semantic drift.**  
The feedback mechanism itself is doing real work, not just the baseline anchor.

**Current limitation:**  
The observer is a cosine similarity proxy between PDE state and goal
projection. This is a 32-dimensional crude signal. The correct observer
is Nelson's SAD divergence signal, which measures geometric drift in
the full message-passing structure.

**This is not yet a full control loop.**  
The correction is single-step and reactive. It has no memory of
trajectory, no drift rate estimation, and operates inside the state
update rather than feeding back to the Gater routing decisions.

## Next Steps

### 1. Replace the proxy observer with Nelson's SAD signal
The current alignment measure is cosine similarity between the PDE state
and a crude goal projection. Nelson's SAD probe measures geometric drift
in the message-passing structure directly — a richer, more sensitive signal.

Integration point is one line:
```python
# Current (proxy):
alignment = cosine_similarity(pde_state, goal_proj).mean().item()

# Target (SAD observer):
alignment = 1.0 - sad_probe.current_divergence(hop_label)
```

### 2. Add temporal memory to the controller
The current correction fires on instantaneous alignment. A real controller
needs drift rate — is alignment getting worse, stable, or recovering?

```python
drift_rate = prev_alignment - current_alignment
alpha = f(current_alignment, drift_rate)
prev_alignment = current_alignment
```

### 3. Feed correction back to the Gater
Currently correction happens inside the state update — a local fix.
Persistent misalignment should feed back to the Gater and change which
channels are active. This is the difference between a state correction
and a routing correction.

Architecture target:
```
SAD observer → drift signal → adaptive controller
                                    ↓              ↓
                             state correction   Gater routing update
```

### 4. Wire the Gater
The Gater validation is complete. Goal steering is confirmed (0.0129
mean pairwise cosine distance across 15 goal perturbations). The Gater
signal will propagate through the CD PDE state update.

The integration question is now: does the Gater's routing geometry
translate into controllable, task-relevant behavior once the SAD
observer closes the loop?

### 5. Trained weight validation
All results to date are on random weights with synthetic data.
The system demonstrates geometric structure, not yet task-level intelligence.
The path from better dynamics to better intelligence requires trained
weights and a real task signal.

---
*The problem is no longer: can we detect drift, or can we stabilize the system.*  
*It is: how do we use the observer signal to drive correction in a principled way.*